In [6]:
import pickle
import pandas as pd

In [2]:
with open("model/model.pkl", "rb") as f:
    model = pickle.load(f)

In [4]:
type(model)
model

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['account_type', 'industry']),
                                                 ('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['employee_count',
                                                   'intent_score',
                                                   'mql_count_90d',
                                                   'trial_started',
                                                   'trial_active_users',
                                                   'web_touchpoints_90d',
                                                   'sales_contacts_90d'])])),
                ('clf',
                 GradientBoostingClassifier(learning_rate=0.05, max_depth=2,
                                            min_samples_leaf=20,
                                            n_estimators=40, random_state=42,
                                            subsample=0.7))])

In [7]:
training_data = pd.read_csv("data/training_data.csv")

In [9]:
training_data.head()

,account_id,account_type,snapshot_date,employee_count,industry,intent_score,mql_count_90d,trial_started,trial_active_users,web_touchpoints_90d,sales_contacts_90d,converted_within_90d
0,ACC-00002,Suspect,2026-07-07,21,Retail,NaN,2,1,1,9,4,0
1,ACC-00003,Suspect,2024-11-27,21,Professional Services,30.9,0,1,1,8,0,0
2,ACC-00004,Prospect,2025-01-18,72,Financial Services,NaN,4,0,0,2,0,0
3,ACC-00005,Prospect,2026-06-22,54,Manufacturing,22.7,0,0,0,3,0,0
4,ACC-00006,Former Customer,2026-01-26,63,Software,NaN,2,0,0,8,0,0


In [10]:
training_data.columns

Index(['account_id', 'account_type', 'snapshot_date', 'employee_count',
       'industry', 'intent_score', 'mql_count_90d', 'trial_started',
       'trial_active_users', 'web_touchpoints_90d', 'sales_contacts_90d',
       'converted_within_90d'],
      dtype='object')

In [12]:
print(training_data.shape)
print(training_data.info())

(1200, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   account_id            1200 non-null   object 
 1   account_type          1200 non-null   object 
 2   snapshot_date         1200 non-null   object 
 3   employee_count        1200 non-null   int64  
 4   industry              1200 non-null   object 
 5   intent_score          718 non-null    float64
 6   mql_count_90d         1200 non-null   int64  
 7   trial_started         1200 non-null   int64  
 8   trial_active_users    1200 non-null   int64  
 9   web_touchpoints_90d   1200 non-null   int64  
 10  sales_contacts_90d    1200 non-null   int64  
 11  converted_within_90d  1200 non-null   int64  
dtypes: float64(1), int64(7), object(4)
memory usage: 112.6+ KB
None


In [18]:
print(training_data["converted_within_90d"].value_counts(normalize=True))
print(training_data["account_type"].value_counts())
print(training_data["industry"].value_counts().head(10))

print("\nMissingness:")
print(training_data.isna().mean().sort_values(ascending=False))

print("\nNumeric summary:")
print(training_data.describe().T)

converted_within_90d
0    0.935
1    0.065
Name: proportion, dtype: float64
account_type
Prospect           632
Suspect            402
Former Customer    166
Name: count, dtype: int64
industry
Software                 213
Manufacturing            210
Retail                   201
Professional Services    201
Healthcare               196
Financial Services       179
Name: count, dtype: int64

Missingness:
intent_score            0.401667
account_id              0.000000
account_type            0.000000
snapshot_date           0.000000
employee_count          0.000000
industry                0.000000
mql_count_90d           0.000000
trial_started           0.000000
trial_active_users      0.000000
web_touchpoints_90d     0.000000
sales_contacts_90d      0.000000
converted_within_90d    0.000000
dtype: float64

Numeric summary:
                       count        mean         std  min   25%   50%    75%  \
employee_count        1200.0  121.140000  213.173731  3.0  29.0  67.0  139.0   
inte

In [19]:
target = "converted_within_90d"

print(training_data[target].value_counts())
print(training_data[target].value_counts(normalize=True))

converted_within_90d
0    1122
1      78
Name: count, dtype: int64
converted_within_90d
0    0.935
1    0.065
Name: proportion, dtype: float64


In [20]:
missing = (
    training_data.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_rate")
)

missing["missing_count"] = training_data.isna().sum()

display(missing)

,missing_rate,missing_count
intent_score,0.401667,482
account_id,0.000000,0
account_type,0.000000,0
snapshot_date,0.000000,0
employee_count,0.000000,0
industry,0.000000,0
mql_count_90d,0.000000,0
trial_started,0.000000,0
trial_active_users,0.000000,0
web_touchpoints_90d,0.000000,0


In [21]:
print(
    "Intent score missing:",
    training_data["intent_score"].isna().mean()
)

Intent score missing: 0.40166666666666667


In [22]:
training_data.groupby(
    training_data["intent_score"].isna()
)[target].agg(["count", "mean"])

,count,mean
intent_score,,
False,718,0.082173
True,482,0.039419


In [ ]:
display(
    training_data.groupby("account_type")[target]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)


,count,mean
account_type,,
Former Customer,166,0.072289
Prospect,632,0.066456
Suspect,402,0.059701


In [25]:
print(training_data["industry"].value_counts())

industry
Software                 213
Manufacturing            210
Retail                   201
Professional Services    201
Healthcare               196
Financial Services       179
Name: count, dtype: int64


In [26]:
industry_conversion = (
    training_data.groupby("industry")[target]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

display(industry_conversion)

,count,mean
industry,,
Manufacturing,210,0.076190
Financial Services,179,0.067039
Healthcare,196,0.066327
Software,213,0.065728
Professional Services,201,0.059701
Retail,201,0.054726


In [28]:
accounts = pd.read_csv("data/accounts_to_score.csv")

print(accounts.shape)
display(accounts.head())
accounts.info()

(300, 11)


,account_id,account_type,snapshot_date,employee_count,industry,intent_score,mql_count_90d,trial_started,trial_active_users,web_touchpoints_90d,sales_contacts_90d
0,ACC-01073,Prospect,2026-04-11,9,Healthcare,NaN,1,0,0,2,0
1,ACC-00533,Prospect,2026-07-19,65,Manufacturing,63.0,1,0,0,7,0
2,ACC-01319,Prospect,2025-08-15,17,Retail,NaN,0,0,0,5,0
3,ACC-00160,Suspect,2026-04-06,63,Professional Services,6.9,0,0,0,8,3
4,ACC-00275,Suspect,2026-04-05,80,Healthcare,46.3,0,0,0,0,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   account_id           300 non-null    object 
 1   account_type         300 non-null    object 
 2   snapshot_date        300 non-null    object 
 3   employee_count       300 non-null    int64  
 4   industry             300 non-null    object 
 5   intent_score         184 non-null    float64
 6   mql_count_90d        300 non-null    int64  
 7   trial_started        300 non-null    int64  
 8   trial_active_users   300 non-null    int64  
 9   web_touchpoints_90d  300 non-null    int64  
 10  sales_contacts_90d   300 non-null    int64  
dtypes: float64(1), int64(6), object(4)
memory usage: 25.9+ KB


In [30]:
display(
    accounts.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_rate")
)

,missing_rate
intent_score,0.386667
account_id,0.000000
account_type,0.000000
snapshot_date,0.000000
employee_count,0.000000
industry,0.000000
mql_count_90d,0.000000
trial_started,0.000000
trial_active_users,0.000000
web_touchpoints_90d,0.000000


In [34]:
import joblib

model = joblib.load("model/model.pkl")

print("Classes:", model.classes_)

feature_cols = [
    "account_type",
    "employee_count",
    "industry",
    "intent_score",
    "mql_count_90d",
    "trial_started",
    "trial_active_users",
    "web_touchpoints_90d",
    "sales_contacts_90d",
]

X = accounts[feature_cols]

proba = model.predict_proba(X)

print(proba.shape)

Classes: [0 1]
(300, 2)


In [36]:
proba

array([[0.95911942, 0.04088058],
       [0.87805934, 0.12194066],
       [0.95911942, 0.04088058],
       [0.92676419, 0.07323581],
       [0.95062034, 0.04937966],
       [0.94990397, 0.05009603],
       [0.91315353, 0.08684647],
       [0.91244087, 0.08755913],
       [0.95042416, 0.04957584],
       [0.94980891, 0.05019109],
       [0.90498113, 0.09501887],
       [0.96011075, 0.03988925],
       [0.95788366, 0.04211634],
       [0.94923755, 0.05076245],
       [0.95082668, 0.04917332],
       [0.94871461, 0.05128539],
       [0.93595425, 0.06404575],
       [0.90931656, 0.09068344],
       [0.95164515, 0.04835485],
       [0.8975974 , 0.1024026 ],
       [0.92366864, 0.07633136],
       [0.92439361, 0.07560639],
       [0.95911942, 0.04088058],
       [0.95260189, 0.04739811],
       [0.95466271, 0.04533729],
       [0.94362157, 0.05637843],
       [0.95503713, 0.04496287],
       [0.95454653, 0.04545347],
       [0.92441618, 0.07558382],
       [0.92702221, 0.07297779],
       [0.

In [37]:
accounts["conversion_probability"] = proba[:, 1]

accounts["conversion_probability"].describe()

count    300.000000
mean       0.065536
std        0.029803
min        0.038071
25%        0.044023
50%        0.054022
75%        0.077973
max        0.209393
Name: conversion_probability, dtype: float64

In [39]:
for pct in [0.10, 0.20, 0.30, 0.50]:
    threshold = accounts["conversion_probability"].quantile(1 - pct)
    
    print(
        f"Top {int(pct*100)}%: "
        f"{(accounts['conversion_probability'] >= threshold).sum()} accounts, "
        f"threshold = {threshold:.4f}"
    )

Top 10%: 30 accounts, threshold = 0.1063
Top 20%: 60 accounts, threshold = 0.0874
Top 30%: 90 accounts, threshold = 0.0731
Top 50%: 151 accounts, threshold = 0.0540
